# Session 3 Homework · Encode, Split, and Run a Model

**Expected time:** 30–45 minutes · **Bring questions** — next session opens with homework review.

You'll take a fresh dataset of laptop listings all the way to a working (black-box) model, using exactly the moves from class.

**Done means:**

- all five exercises attempted
- Exercise 2: the encoded table has **no text columns left**
- Exercise 3: you print **both** a training score and a test score from `.score()`
- Exercise 4: you print a predicted price next to the actual price for one held-out laptop
- Exercise 5: a reflection of **at least three sentences** using *memorizing* and *learning* correctly, plus one sentence on when scaling would matter

The model is a black box again — we run the three moves and read the result. How it works inside is Module 2.

## Exercise 1 · Load and find the text column

Run the cell to build the dataset, then look at it.

In [ ]:
import pandas as pd
import numpy as np

# A reproducible dataset of laptop listings (fixed seed -> same data every run).
rng = np.random.default_rng(3)
n = 60
brands = np.array(["Acer", "Dell", "Apple", "Lenovo"])

brand = rng.choice(brands, size=n)
ram_gb = rng.choice([8, 16, 32], size=n)
ssd_gb = rng.choice([256, 512, 1024], size=n)
screen_inch = np.round(rng.uniform(13, 17, size=n), 1)
weight_kg = np.round(rng.uniform(1.0, 2.4, size=n), 2)

brand_premium = np.select(
    [brand == "Apple", brand == "Dell"],
    [25.0, 5.0],
    default=0.0,
)
price_thousands = np.round(
    15 + ram_gb * 0.9 + ssd_gb * 0.03 + brand_premium
    + screen_inch * 0.5 - weight_kg * 4
    + rng.normal(0, 3, size=n),
    1,
)

laptops = pd.DataFrame({
    "brand": brand,
    "ram_gb": ram_gb,
    "ssd_gb": ssd_gb,
    "screen_inch": screen_inch,
    "weight_kg": weight_kg,
    "price_thousands": price_thousands,
})
laptops.head()

In [ ]:
# ✏️ TODO: take a structured look. (These already work.)
print("shape:", laptops.shape)
laptops.info()

✏️ Which single column is **text** (the one a model can't do arithmetic on)?

*Your answer:* 

## Exercise 2 · One-hot encode the text column

In [ ]:
# ✏️ TODO: one-hot encode the `brand` column with dtype=int. (Working version provided.)
laptops_encoded = pd.get_dummies(laptops, columns=["brand"], dtype=int)

laptops_encoded.head()

In [ ]:
# Proof there are no text columns left: this should be an empty list.
text_columns_left = list(laptops_encoded.select_dtypes(include="object").columns)
print("text columns remaining:", text_columns_left)

✏️ Why is one-hot encoding `brand` more honest than numbering the brands 0, 1, 2, 3?

*Your answer:* 

## Exercise 3 · Split, then run the three moves

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# ✏️ TODO: build X (all features) and y (the label = price_thousands).
y = laptops_encoded["price_thousands"]
X = laptops_encoded.drop(columns=["price_thousands"])

# Hide 20% as a test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = LinearRegression()
model.fit(X_train, y_train)

print("training score:", round(model.score(X_train, y_train), 3))
print("test score:    ", round(model.score(X_test, y_test), 3))

✏️ Report your test score. Was the training score higher than the test score? Why does that make sense?

*Your answer:* 

## Exercise 4 · Predict one held-out laptop

In [ ]:
# Predict the price of the FIRST laptop in the hidden test set, and compare to the truth.
first_prediction = model.predict(X_test.iloc[[0]])[0]
first_actual = y_test.iloc[0]

print(f"predicted: {first_prediction:.1f} thousand")
print(f"actual:    {first_actual:.1f} thousand")
print(f"off by:    {abs(first_prediction - first_actual):.1f} thousand")

✏️ Is the prediction close? In one line: would you trust a single prediction more, or a score across many held-out laptops?

*Your answer:* 

## Exercise 5 · Why hide data? (open-ended)

✏️ Write **at least three sentences**, using *memorizing* and *learning* correctly:

Why did we score the model on laptops it never saw during training, instead of on the laptops it learned from? Connect it to the Session 1 model that memorized an answer key and scored a perfect 100%.

Then add **one sentence**: name a kind of model (or situation) where you *would* need to scale the features first.

*Your reflection:*

…